# APEDP-X / CPMD: End-to-End Scientific Discovery & Epistemic Referee Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/psalmprax/science/blob/master/notebooks/apedp_scientific_discovery_tutorial.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/psalmprax/science/blob/master/notebooks/apedp_scientific_discovery_tutorial.ipynb)

Welcome to **APEDP-X (Automated Physics & Epistemic Discovery Platform - eXtended)**.
This interactive tutorial demonstrates the complete scientific discovery pipeline governed by strict epistemic refereeing:

1. **Universal Environment & Storage Adapter** (`apedp_dyn.notebook_adapter`)
2. **Observational Data Simulation** (Microstructure orderflow arrivals & price impact)
3. **Candidate Hypothesis Generation** (`HawkesOrderFlowProposer` & `SparseMicrostructureProposer`)
4. **Stage 1-6 Epistemic Referee Adjudication** (Novelty, Diebold-Mariano incrementality, Deflated Sharpe Ratio)
5. **Anti-Snooping Holdout Discipline** (`FullLineageExperimentLedger` & slice burning)
6. **Remote Storage Synchronization & Immutable Snapshots** (`remote_storage`)
7. **Micro-Sample Symbolic Discovery** with the structure-aware, graph-validated referee
   (RimeProposer on Langmuir 1918, N=11; frozen record 2026-10)
8. **Companion Pathways** — SIR epidemiology discovery (`CompartmentalSirProposer`, exact mass
   conservation) and the Michaelis–Menten pathway through the same graph-validated referee


## Step 1: Universal Runtime Adapter Initialization
We initialize the runtime adapter to detect the platform (Google Colab, Kaggle, Jupyter, or Local Workstation), configure temporary scratch storage, and verify holdout persistence safeguards.

In [ ]:
import sys, os
from pathlib import Path

from apedp_dyn.notebook_adapter import init_environment, temporary_storage_scope

# Initialize environment (auto-detects Colab, Kaggle, Jupyter, or Local)
config = init_environment(
    mount_gdrive=False,              # Set True in Google Colab to preserve holdout ledgers across VM recycles
    temp_dir="/tmp/apedp_scratch",   # High-speed scratch storage for intermediate computations
    auto_compile_kernel=False        # Uses bit-accurate pure-Python fallback if Rust toolchain is absent
)

print(config.summary())
config  # Renders rich interactive HTML badge in Jupyter / Colab cells

## Step 2: Synthetic Observational Microstructure Data
We simulate high-frequency trade arrival point processes (Hawkes self-exciting intensity) and mid-price series for in-sample discovery, validation, and confirmation holdout adjudication.

In [ ]:
import numpy as np

np.random.seed(42)
n_steps = 2000
dt = 0.1

# Simulate self-exciting Hawkes order arrival intensities
base_intensity = 0.5
intensities = np.zeros(n_steps)
arrivals = np.zeros(n_steps)
curr_intensity = base_intensity

for t in range(n_steps):
    arrivals[t] = np.random.poisson(curr_intensity * dt)
    curr_intensity = base_intensity + 0.8 * (curr_intensity - base_intensity) + 0.5 * arrivals[t]
    intensities[t] = curr_intensity

# Simulate mid-price random walk with flow impact
noise = np.random.normal(0, 0.05, n_steps)
price_changes = 0.02 * (arrivals - np.mean(arrivals)) + noise
prices = 100.0 + np.cumsum(price_changes)

print(f"Generated {n_steps} microstructure steps. Final Price: {prices[-1]:.2f}, Mean Arrival Rate: {np.mean(arrivals):.2f}")

## Step 3: Candidate Hypothesis Generation (Proposers)
Per `AGENTS.md §1.1`, candidate hypotheses are strictly emitted by independent Proposers and never decide their own acceptance.

In [ ]:
import pandas as pd
from apedp_dyn.proposers.hawkes_orderflow_proposer import HawkesOrderFlowProposer
from apedp_dyn.proposers.sparse_microstructure_proposer import SparseMicrostructureProposer

hawkes_proposer = HawkesOrderFlowProposer(decay_beta=1.5, intensity_threshold_quantile=0.80)
micro_proposer = SparseMicrostructureProposer(l1_ratios=[0.5], alphas=[0.001])

# Prepare observational inputs for proposers
arrival_timestamps = np.cumsum(0.1 + np.abs(price_changes[:500]))
hawkes_candidates = hawkes_proposer.fit_and_propose(arrival_timestamps)

df_micro = pd.DataFrame({
    "price": prices[:500],
    "V": np.abs(arrivals[:500]) * 1000.0 + 100.0,
    "taker_buy": np.abs(arrivals[:500]) * 550.0 + 50.0,
})
micro_candidates = micro_proposer.fit_and_propose({"BTC": df_micro})

print(f"Hawkes Proposer emitted {len(hawkes_candidates)} candidate hypotheses:")
for c in hawkes_candidates[:2]:
    print(f"  • [{c.candidate_id}] Archetype: {c.strategy_archetype.value} | Formula: {c.formula}")

print(f"\nSparse Microstructure Proposer emitted {len(micro_candidates)} candidate hypotheses:")
for c in micro_candidates[:2]:
    print(f"  • [{c.candidate_id}] Archetype: {c.strategy_archetype.value} | Formula: {c.formula}")


## Step 4: Multi-Stage Epistemic Referee Adjudication (Gates 1–6)
We adjudicate the proposals across the canonical preregistered epistemic gates:
- **Gate 4**: Predictive incrementality via Diebold-Mariano test ($p < 0.01$)
- **Gate 5**: Deflated Sharpe Ratio (DSR $\ge 0.95$) accounting for selection bias across multiple trials

In [ ]:
from apedp_dyn.epistemic.novelty_adjudication import evaluate_predictive_incrementality
from apedp_dyn.epistemic.trial_ledger import compute_deflated_sharpe_ratio

# Gate 4: Diebold-Mariano Predictive Incrementality
y_true = price_changes[1000:1500]
y_base = y_true + 0.8 * np.random.randn(len(y_true))
y_cand = y_true + 0.2 * np.random.randn(len(y_true))

dm_res = evaluate_predictive_incrementality(y_true, y_cand, y_base, alpha=0.01)
print(f"Gate 4 (Diebold-Mariano): Stat={dm_res.dm_statistic:.3f}, p-value={dm_res.p_value:.4e} -> Verdict: {dm_res.verdict}")

# Gate 5: Deflated Sharpe Ratio under search multiplicity
dsr_result = compute_deflated_sharpe_ratio(
    observed_sharpe=1.65,
    n_trials=50,
    var_trials=0.15,
    n_bars=500
)
dsr = dsr_result["dsr"]
dsr_status = "PASS" if dsr >= 0.95 else "FAIL"
print(f"Gate 5 (Deflated Sharpe Ratio): DSR={dsr:.4f} -> {dsr_status}")


## Step 5: Anti-Snooping Holdout Discipline (`AGENTS.md §1.2`)
Confirmation holdouts are consumable, scarce resources. If a candidate hypothesis fails out-of-sample confirmation, that `(dataset_id, slice_id)` slice is permanently burned in `FullLineageExperimentLedger`. Any subsequent attempt to reuse that slice triggers `HoldoutContaminationError`.

In [ ]:
from pathlib import Path

from apedp_dyn.epistemic.experiment_ledger import (
    FullLineageExperimentLedger,
    CandidateLifecycleState,
    DatasetRole,
    HoldoutContaminationError,
)

# -- DEMO-vs-PRODUCTION marker -----------------------------------------------
# This entire ledger is a SYNTHETIC DEMO namespace (fixed ids cand_alpha_01 /
# cand_beta_02, slice "holdout_slice_Q3"). It carries no real discovery
# records, so when a previous run of this notebook already burned the demo
# slice at the default path, we recreate it and the demo stays re-runnable
# in a fresh kernel. Production holdout ledgers live under different
# filenames and are NEVER touched here; AGENTS.md Section 1.2 anti-snooping
# discipline applies fully to those real ledgers.
demo_ledger_path = Path(config.get_ledger_path("tutorial_experiment_ledger.jsonl"))
if demo_ledger_path.exists():
    demo_ledger_path.unlink()
    print(f"[demo-reset] Recreated burned demo ledger at: {demo_ledger_path}")

ledger = FullLineageExperimentLedger(
    persistence_path=demo_ledger_path
)

# Register Candidate Alpha
ledger.register_candidate(
    candidate_id="cand_alpha_01",
    proposer_id="hawkes_orderflow",
    formula="alpha_model(flow, dt)"
)

# Expose Candidate Alpha to Confirmation Holdout Slice
ledger.record_data_exposure(
    candidate_id="cand_alpha_01",
    dataset_id="microstructure_2026",
    slice_identifier="holdout_slice_Q3",
    role=DatasetRole.CONFIRMATION_HOLDOUT,
    outcome_status="FAILED"
)

# Candidate Alpha fails confirmation and is rejected
ledger.update_lifecycle_state(
    candidate_id="cand_alpha_01",
    new_state=CandidateLifecycleState.CONFIRMATION_REJECTED,
    rejection_reason="Holdout RMSE exceeded bounds"
)

print("Burned Holdout Registry:", ledger.get_burned_holdouts())

# Verify Anti-Snooping Invariant: Candidate Beta attempts to reuse the burned slice
ledger.register_candidate(
    candidate_id="cand_beta_02",
    proposer_id="microstructure",
    formula="beta_model(spread, vol)"
)

try:
    ledger.record_data_exposure(
        candidate_id="cand_beta_02",
        dataset_id="microstructure_2026",
        slice_identifier="holdout_slice_Q3",
        role=DatasetRole.CONFIRMATION_HOLDOUT
    )
    print("ERROR: Holdout contamination check failed to trigger!")
except HoldoutContaminationError as exc:
    print(f"\n[SUCCESS] Anti-snooping firewall triggered as expected:\n{exc}")

## Step 6: Scoped Scratch Workspaces & Immutable Snapshots
We demonstrate `temporary_storage_scope()` for fast, isolated intermediate data with guaranteed cleanup, and `export_snapshot()` for auditable persistence.

In [ ]:
# 1. Scoped temporary scratch workspace
with temporary_storage_scope() as scratch_dir:
    scratch_file = scratch_dir / "intermediate_sweep.dat"
    scratch_file.write_text("intermediate tensor sweep state")
    print(f"Scratch file active at: {scratch_file} (exists: {scratch_file.exists()})")

print(f"After exiting context, scratch directory is safely purged: {scratch_file.exists()}")

# 2. Export immutable timestamped ledger snapshot
snapshot_path = ledger.export_snapshot()
print(f"\nExported immutable ledger snapshot:\n  {snapshot_path}")

## Step 7: Micro-Sample Symbolic Discovery with the Structure-Aware, Graph-Validated Referee

The current state of the platform (2026-10 frozen record,
`benchmarks/rime_microsample_adjudication_results.json`) adds an autonomous
**symbolic mechanism search** for micro-samples: `RimeProposer`'s Rust beam search
selects a saturating structure on the Langmuir 1918 nitrogen isotherm (N = 11)
from a declared grammar — and an independent referee then validates the
**declared structure** (not intercept proxies): the exact family certificate
$\mathrm{d}q/\mathrm{d}p = w\,s/(s+p)^2 > 0$, gates $\{w > 0,\ q_\infty = w+b > 0,$
exact monotonicity$\}$, and the origin invariant $q(0) = b$ reported explicitly
with an `origin_strict_satisfied` flag (no $|b|$ tolerance gates).

In [ ]:
import pandas as pd
from apedp_dyn.proposers.rime_proposer import RimeProposer

# Langmuir (1918) Table VII — nitrogen on mica at 90 K
# p in historical 'Langmuir bars' (barye = 0.1 Pa), uptake q in mm^3 at 20 deg C, 760 mm
df_langmuir = pd.DataFrame({
    "p": [34.0, 23.8, 17.3, 13.0, 9.5, 7.4, 6.1, 5.0, 4.0, 3.4, 2.8],
    "q": [33.0, 30.8, 28.2, 25.5, 23.9, 21.6, 19.0, 17.0, 15.1, 13.4, 12.0],
})

# min_rows must be lowered from the production safeguard (30) for this N=11 sample
# -- an explicitly disclosed exploratory override, per the frozen record
rime = RimeProposer(
    feature_cols=["p"], target_col="q", static_mode=True,
    algebraic_transforms=["sq", "ssqrt", "slog", "sat_norm"],
    min_rows=5, max_proposals=2, seed=42,
)
rime_candidates = rime.fit_and_propose(df_langmuir)

print(f"RimeProposer emitted {len(rime_candidates)} candidates "
      "(autonomous structure search over quadratic / sqrt / log / saturation grammar):")
for c in rime_candidates:
    p = c.parameters
    print(f"  - [{c.candidate_id[:24]}] {c.parameters['identifiability_status']}: "
          f"{c.formula}  (dev R2={p['dev_r2']:.4f})")
print(f"\nTop-rank structure: saturating rational family sat_norm(p)")

In [ ]:
import numpy as np

# The frozen adjudication harness is importable at repo root
from benchmarks.run_rime_microsample_adjudication import (
    adjudicate_structure_aware,
    certify_origin_constrained_rime,
    sound_monotonicity_v2,
)

winner = rime_candidates[0]
w_fit = float(winner.parameters["scale"])
b_fit = float(winner.parameters["intercept"])
s_dev = float(np.std(df_langmuir["p"].values))

# 1. Exact family certificate: dq/dp = w*s/(s+p)^2 > 0 (holds for ANY intercept b)
cert, cert_why = certify_origin_constrained_rime(w_fit, s_dev)
print(f"Exact family certificate monotonicity: {cert}")

# 2. General algebraic sign certification (exact real-root isolation, no tolerance filter)
import sympy as sp
p_sym = sp.Symbol("p", positive=True, real=True)
v2, v2_why = sound_monotonicity_v2(w_fit * (p_sym / s_dev) / (1 + p_sym / s_dev) + b_fit, p_sym)
print(f"sound_monotonicity_v2 verdict:      {v2}")

# 3. Structure-aware gates with the origin invariant reported, never silently tolerated
verdict, reports, referee_why = adjudicate_structure_aware([
    {"proposal": winner, "s": s_dev},
])
rep = reports[0]
print(f"Structure-aware referee verdict:    {verdict} ({len(reports)}/{len(reports)} entries)")
print(f"  gates: {rep['gates']}")
print(f"  q(0) = {rep['q0']:+.4f}  -> origin_strict_satisfied = {rep['origin_strict_satisfied']} "
      "(strict-law promotion stays reserved for the b=0 constrained baseline)")
print(f"  {referee_why}")

### The frozen 2026-10 record this step reproduces

Measured in `benchmarks/rime_microsample_adjudication_results.json` (audit-only):

| Quantity | Measured |
|---|---|
| Langmuir 8-scale origin-constrained LOOCV RMSE | **0.404810 mm** (continuous Langmuir fit baseline: 0.408720) |
| Per-fold `RimeProposer` over the Python-built bank, bank-restricted | RMSE **0.482436**, saturation family at top rank in 11/11 folds |
| Per-fold `RimeProposer` under the *unrestricted* declared grammar (kernel-derived scales) | property recovery **11/11 folds**, RMSE 0.629997 |
| Graph-equation structural validation (metadata + executable affine-readout check) | 11/11 folds validated, max deviation **0.0** (machine-exact) |
| Michaelis-Menten pathway (same referee) | **12/12 folds PROMOTED**, actual-RIME LOOCV RMSE 0.1316; Vmax recovered within 5% |
| Negative controls (5 seeds x noise, decay) | **0/20 false promotions** under both gate semantics |

Every number above is frozen in the artifact and pinned by
`tests/test_dyn_rime_microsample_adjudication.py` (24 regression tests).

## Step 8: Companion Discovery Pathways — SIR Epidemiology & Michaelis–Menten Kinetics

The same proposer/referee separation extends beyond the Langmuir isotherm:

- **SIR epidemiology (live discovery)**: `CompartmentalSirProposer` fits transmission
  and recovery rates from simulated Kermack–McKendrick dynamics while enforcing the
  exact conserved-quantity invariant $S(t)+I(t)+R(t) = N$ (mass conservation), and
  emits a declared `DYNAMICAL_SYSTEM` candidate carrying $R_0 = \beta/\gamma$ in metadata.
- **Michaelis–Menten (frozen pathway)**: the same enzyme-kinetic saturating structure
  $v = V_\mathrm{max}\,S/(K_m+S)$ runs through the *graph-validated* structure-aware
  referee of Step 7. The frozen record (seed-pinned, audit-only harness) measures
  **12/12 folds PROMOTED** with actual-RIME LOOCV RMSE **0.1316** — reproduced live below.


In [ ]:
import numpy as np
import pandas as pd
from apedp_dyn.proposers.stem_proposers import CompartmentalSirProposer

# Simulate Kermack-McKendrick SIR with known truth: beta=0.3, gamma=0.12, R0=2.5
n_pop, dt, n_steps = 1_000_000.0, 1.0, 80
beta_true, gamma_true = 0.3, 0.12
S, I, R = np.empty(n_steps), np.empty(n_steps), np.empty(n_steps)
S[0], I[0], R[0] = 0.99 * n_pop, 0.01 * n_pop, 0.0
for t in range(1, n_steps):
    dS = -beta_true * S[t-1] * I[t-1] / n_pop * dt
    dI = (beta_true * S[t-1] * I[t-1] / n_pop - gamma_true * I[t-1]) * dt
    S[t], I[t], R[t] = S[t-1] + dS, I[t-1] + dI, R[t-1] - dS - dI

sir = CompartmentalSirProposer(dt=dt, max_pop_drift=0.02).fit_and_propose(
    pd.DataFrame({"S": S, "I": I, "R": R})
)
if sir:
    p = sir[0].parameters
    print(f"SIR discovery: {sir[0].formula}")
    print(f"  fitted beta = {p['beta']:.4f} (true 0.30)")
    print(f"  fitted gamma = {p['gamma']:.4f} (true 0.12)")
    print(f"  fitted R0 = {p['R0']:.3f} (true 2.50)")
    print(f"  conserved population N = {p['population_N']:.0f}; "
          f"metadata domain: {sir[0].metadata['domain']}")
else:
    print("SIR proposer ABSTAINED (mass-conservation or support-size gate tripped)")


In [ ]:
from benchmarks.run_rime_microsample_adjudication import run_michaelis_menten_graph_validated

# Companion pathway frozen record (audit-only harness, seed-pinned, deterministic):
# v = Vmax * S / (Km + S), true Vmax = 10, Km = 2.5, N = 12, noise sigma 0.12.
mm = run_michaelis_menten_graph_validated(seed=42)

print(f"MM graph-validated pathway: {mm['promoted']}/{mm['n_folds']} folds PROMOTED "
      f"(structure-validated {mm['structure_validated_folds']}/{mm['n_folds']}, "
      f"abstentions {mm['abstentions']})")
print(f"Actual-RIME LOOCV RMSE = {mm['loocv_rmse_actual_rime']:.4f} "
      "(Langmuir companion record: 0.404810 mm)")

fold0 = next(f for f in mm["folds"] if not f["abstained"])
print(f"Fold 0 winner: {fold0['formula']}")
print(f"  declared s_m = {fold0['declared_s_m']:.4f}, w = {fold0['w']:.4f}, "
      f"b = {fold0['b']:.4f} -> q_inf (implied Vmax) = {fold0['q_inf']:.4f}")
print(f"  graph validated: {fold0['structure_validated']}, "
      f"origin_strict_satisfied: {fold0['origin_strict_satisfied']} (b=0 by construction)")

t = mm["tournament"]
print(f"Full-bank tournament: {t['referee_verdict']} "
      f"({t['admissible']}/{t['n_proposals']} admissible, top family = {t['top_rank_family']})")
